In [19]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
import asyncio
import uvicorn
import sqlite3

In [20]:
# Database initalization

# conn = sqlite3.connect("todo_db.db", check_same_thread=False)
# conn.row_factory = sqlite3.Row

def connect_db():
   conn = sqlite3.connect("user_db.db")
   conn.row_factory = sqlite3.Row
   return conn

def convert_to_json(data):
  return [dict(res) for res in data]

def rows_to_dict(rows):
    return [dict(row) for row in rows]


In [21]:
def get_users():
    conn = connect_db()
    rows = conn.execute("SELECT * FROM user").fetchall()
    conn.close()
    return rows_to_dict(rows)

def get_users_by_id(id):
    conn = connect_db()
    data = conn.execute(f"SELECT * FROM user WHERE id = {id}").fetchone()
    conn.close()
    if data is None:
        raise HTTPException(status_code=404, detail="User not found.")
    return dict(data)

In [ ]:
from pydantic import BaseModel
from typing import Optional

class User(BaseModel):
  name: Optional[str] = None
  email: Optional[str] = None
  tel: Optional[str] = None

In [23]:
from fastapi.staticfiles import StaticFiles

app = FastAPI()
app.mount("/static", StaticFiles(directory="static"), name="static")
templates = Jinja2Templates(directory="templates")

In [28]:
# HTML Endpoints (Routes)
@app.get("/", response_class=HTMLResponse)
def root():
  return """

  <html>
    <body>
      <a href="/docs">docs</a>
    </body>
  </html>

  """

@app.get("/users", response_class=HTMLResponse)
def users_html(request: Request):
  return templates.TemplateResponse(
    "users.html",
    {
      "request": request,
      "users": get_users()
    }
  )

@app.get("/users/{id}/edit", response_class=HTMLResponse)
def edit_user_html(request: Request, id: int):
  user = get_users_by_id(id)
  return templates.TemplateResponse(
    "edit_user.html",
    {
      "request": request,
      "user": user
    }
  )

@app.get("/users/create", response_class=HTMLResponse)
def create_user_html(request: Request):
  return templates.TemplateResponse(
    "create_user.html",
    {
      "request": request
    }
  )

In [ ]:
# API Endpoints (Routes)

# Read
@app.get("/api/users")
def users():
  return {
    "data": get_users()
  }

@app.get("/api/users/{id}")
def users_by_id(id):
  return {
    "data": get_users_by_id(id)
  }

# Create
@app.post("/api/users")
def create_user(user: User):
  conn = connect_db()
  allowed_fields = {"name", "email", "tel"}
  incoming_data = user.model_dump()

  safe_data = {}
  for key, value in incoming_data.items():
    if key in allowed_fields:
      safe_data[key] = value
    
  if not safe_data:
    conn.close()
    raise HTTPException(status_code=400, detail="No valid fields to insert.")
  
  columns = ", ".join(safe_data.keys())
  placeholders = ", ".join([f":{key}" for key in safe_data.keys()])

  sql = f"INSERT INTO user ({columns}) VALUES ({placeholders})"

  cursor = conn.execute(sql, safe_data)
  conn.commit()

  new_id = cursor.lastrowid
  new_data = conn.execute(
    f"SELECT * FROM user WHERE id = :id", {"id": new_id}
  ).fetchone()
  conn.close()

  return dict(new_data)

# Update
@app.put("/api/users/{id}")
def update_user(id: int, user: User):
  conn = connect_db()
  row = conn.execute(
    f"SELECT * FROM user WHERE id = {id}"
  ).fetchone()

  if row is None:
    conn.close()
    raise HTTPException(status_code=404, detail="User not found.")
  
  allowed_fields = {"name", "email", "tel"}
  # incoming_data = user.dict(exclude_unset=True)
  incoming_data = user.model_dump(exclude_unset=True)

  safe_data = {}
  for key, value in incoming_data.items():
    if key in allowed_fields:
      safe_data[key] = value
    
  if not safe_data:
    conn.close()
    raise HTTPException(status_code=400, detail="No valid fields to update.")
  
  set_parts, values = [], {"id": id}
  for key, value in safe_data.items():
    set_parts.append(f"{key} = :{key}")
    values[key] = value
  set_clause = ", ".join(set_parts)

  sql = f"UPDATE user SET {set_clause} WHERE id = :id"

  conn.execute(sql, values)
  conn.commit()

  updated_data = conn.execute(
    f"SELECT * FROM user WHERE id = :id", {"id": id}
  ).fetchone()
  conn.close()

  return dict(updated_data)

# Delete
@app.delete("/api/users/{id}")
def delete_user(id: int):
  conn = connect_db()
  row = conn.execute(
    f"SELECT * FROM user WHERE id = {id}"
  ).fetchone()

  if row is None:
    conn.close()
    raise HTTPException(status_code=404, detail="User not found.")
  
  conn.execute(
    f"DELETE FROM user WHERE id = {id}"
  )
  conn.commit()
  conn.close()

  return {"detail": "User deleted successfully."}

In [30]:
if __name__ == "__main__":
  config = uvicorn.Config(app)
  server = uvicorn.Server(config)
  await server.serve()

INFO:     Started server process [11440]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52948 - "GET /users HTTP/1.1" 200 OK
INFO:     127.0.0.1:52948 - "GET /static/css/style.css HTTP/1.1" 304 Not Modified
INFO:     127.0.0.1:53271 - "GET /users HTTP/1.1" 200 OK
INFO:     127.0.0.1:53271 - "GET /users/create HTTP/1.1" 200 OK
INFO:     127.0.0.1:52743 - "GET /users/create HTTP/1.1" 200 OK
INFO:     127.0.0.1:64060 - "GET /users/create HTTP/1.1" 200 OK
INFO:     127.0.0.1:53007 - "GET /users HTTP/1.1" 200 OK
INFO:     127.0.0.1:53007 - "GET /users/create HTTP/1.1" 200 OK
INFO:     127.0.0.1:51625 - "POST /api/users/ HTTP/1.1" 307 Temporary Redirect
INFO:     127.0.0.1:51625 - "POST /api/users HTTP/1.1" 200 OK
INFO:     127.0.0.1:51625 - "GET /users HTTP/1.1" 200 OK
INFO:     127.0.0.1:51625 - "GET /users HTTP/1.1" 200 OK
INFO:     127.0.0.1:54393 - "GET /users HTTP/1.1" 200 OK
INFO:     127.0.0.1:54393 - "GET /static/css/style.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:54671 - "GET /users HTTP/1.1" 200 OK
INFO:     127.0.0.1:54671 - "GET /static/css/style

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [11440]
